In [8]:
import os
import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm

In [ ]:
def extract_best_speaker(tsv_path, min_samples=500):
    """
    Finds the speaker with the best balance of high-quality ratings and total samples,
    and returns their specific audio metadata.
    """
    # Load the metadata (using validated.tsv gives you the cleanest baseline)
    df = pd.read_csv(tsv_path, sep='\t')
    
    # Group by client_id to get aggregate statistics per speaker
    speaker_stats = df.groupby('client_id').agg(
        sample_count=('path', 'count'),
        avg_up_votes=('up_votes', 'mean'),
        avg_down_votes=('down_votes', 'mean')
    ).reset_index()
    
    # Keep only speakers who meet your minimum data requirement
    filtered = speaker_stats[speaker_stats['sample_count'] >= min_samples].copy()
    
    if filtered.empty:
        print(f"No speakers found with at least {min_samples} samples.")
        return None, None
        
    # Calculate a quality score: 
    # (Average Upvotes - 2x Average Downvotes) * Total Volume
    filtered['quality_score'] = (
        (filtered['avg_up_votes'] - (filtered['avg_down_votes'] * 2)) 
        * filtered['sample_count']
    )
    
    # Sort to float the absolute best candidate to the top
    best_speakers = filtered.sort_values(by='quality_score', ascending=False)
    top_client_id = best_speakers.iloc[0]['client_id']
    
    # Isolate the original dataframe rows for this top-tier speaker
    top_speaker_data = df[df['client_id'] == top_client_id]
    
    print(f"🏆 Top Speaker ID: {top_client_id}")
    print(f"📊 Total Samples: {best_speakers.iloc[0]['sample_count']}")
    print(f"⭐ Quality Score: {best_speakers.iloc[0]['quality_score']:.2f}")
    
    return top_speaker_data, best_speakers

In [33]:
def filter_xtts_subset(df, target_client_id, target_samples=500):
    """
    Filters a massive single-speaker dataset down to the absolute best 
    samples for XTTSv2 fine-tuning.
    """
    # 1. Isolate the target speaker
    speaker_df = df[df['client_id'] == target_client_id].copy()
    print(f"Initial samples for speaker: {len(speaker_df)}")
    
    # 2. Strict quality control: 0 downvotes, at least 2 upvotes
    strict_quality = speaker_df[
        (speaker_df['down_votes'] == 0) & 
        (speaker_df['up_votes'] >= 2)
    ].copy()
    print(f"Samples after strict voting filter: {len(strict_quality)}")
    
    # 3. Goldilocks length filter (approx. 5 to 15 words)
    # This acts as a proxy for audio duration (roughly 2 to 6 seconds)
    strict_quality['word_count'] = strict_quality['sentence'].apply(lambda x: len(str(x).split()))
    length_filtered = strict_quality[
        (strict_quality['word_count'] >= 5) & 
        (strict_quality['word_count'] <= 15)
    ]
    print(f"Samples after text length filter: {len(length_filtered)}")
    
    # 4. Cap the dataset size
    if len(length_filtered) > target_samples:
        # Random_state ensures reproducibility if you need to run this again
        final_subset = length_filtered.sample(n=target_samples, random_state=42)
        print(f"Randomly sampled down to {target_samples} target samples.")
    else:
        final_subset = length_filtered
        print(f"Kept all {len(final_subset)} samples (below target max).")
        
    return final_subset

In [34]:
def filter_xtts_subset_chinese(df, target_client_id, target_samples=500):
    """
    Filters a massive single-speaker Chinese dataset down to the absolute best 
    samples for XTTSv2 fine-tuning using character length.
    """
    # 1. Isolate the target speaker
    speaker_df = df[df['client_id'] == target_client_id].copy()
    print(f"Initial samples for speaker: {len(speaker_df)}")
    
    # 2. Strict quality control: 0 downvotes, at least 2 upvotes
    strict_quality = speaker_df[
        (speaker_df['down_votes'] == 0) & 
        (speaker_df['up_votes'] >= 2)
    ].copy()
    print(f"Samples after strict voting filter: {len(strict_quality)}")
    
    # 3. Goldilocks length filter (approx. 10 to 30 characters)
    # ⚠️ THE FIX: We use len(str(x)) to count characters, NOT .split()
    strict_quality['char_count'] = strict_quality['sentence'].apply(lambda x: len(str(x)))
    
    length_filtered = strict_quality[
        (strict_quality['char_count'] >= 10) & 
        (strict_quality['char_count'] <= 30)
    ]
    print(f"Samples after text length filter: {len(length_filtered)}")
    
    # 4. Cap the dataset size
    if len(length_filtered) > target_samples:
        final_subset = length_filtered.sample(n=target_samples, random_state=42)
        print(f"Randomly sampled down to {target_samples} target samples.")
    else:
        final_subset = length_filtered
        print(f"Kept all {len(final_subset)} samples (below target max).")
        
    return final_subset

In [9]:
def process_and_convert_to_wav(df, source_audio_dir, output_audio_dir, target_sr=22050):
    """
    Takes a filtered Common Voice dataframe, locates the corresponding .mp3 files,
    converts them to XTTS-ready .wav format (22050Hz, Mono, 16-bit), and saves them.
    """
    # Create the output directory if it doesn't exist
    os.makedirs(output_audio_dir, exist_ok=True)
    
    processed_count = 0
    missing_files = []
    
    print(f"Starting conversion of {len(df)} files to {target_sr}Hz WAV...")
    
    # tqdm gives us a nice progress bar so you aren't staring at a blank terminal
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing Audio"):
        mp3_filename = row['path'] 
        input_path = os.path.join(source_audio_dir, mp3_filename)
        
        # Swap the .mp3 extension for .wav for the new file
        wav_filename = os.path.splitext(mp3_filename)[0] + '.wav'
        output_path = os.path.join(output_audio_dir, wav_filename)
        
        if os.path.exists(input_path):
            try:
                # librosa.load automatically converts to mono and resamples to target_sr!
                audio, sr = librosa.load(input_path, sr=target_sr, mono=True)
                
                # Save as a standard 16-bit PCM WAV file
                sf.write(output_path, audio, target_sr, subtype='PCM_16')
                processed_count += 1
                
            except Exception as e:
                print(f"\nError processing {mp3_filename}: {e}")
        else:
            missing_files.append(mp3_filename)
            
    print(f"\n✅ Successfully processed and converted {processed_count} files.")
    
    if missing_files:
        print(f"⚠️ Warning: Could not find {len(missing_files)} files in the source directory.")
        
    return processed_count

In [39]:
def generate_xtts_metadata(df, output_csv_path):
    """
    Generates a metadata.csv file with your exact requested directory structure.
    Format: full_audio_path|transcript|normalized_transcript
    """
    metadata_lines = []
    
    for index, row in df.iterrows():
        # 1. Swap the .mp3 extension from Common Voice to .wav
        wav_filename = os.path.splitext(row['path'])[0] + '.wav'
        
        # 2. Prepend the exact directory structure you requested
        full_audio_path = f"emoji_project/data/Chinese/wavs/{wav_filename}"
        
        # 3. Clean the text (remove accidental newlines or extra spaces)
        text = str(row['sentence']).replace('\n', ' ').strip()
        
        # 4. Combine using the pipe '|' delimiter. 
        # Column 1: Audio Path
        # Column 2: Transcript
        # Column 3: Normalized Transcript (XTTS handles normalization natively, so we duplicate)
        line = f"{full_audio_path}|{text}|{text}"
        metadata_lines.append(line)
        
    # Write to file
    with open(output_csv_path, 'w', encoding='utf-8') as f:
        for line in metadata_lines:
            f.write(line + '\n')
            
    print(f"📝 Saved {len(metadata_lines)} training entries to: {output_csv_path}")

## French Dataset

In [ ]:
tsv_dir = r"C:\Users\alvar\Code\Github\MedEmoji-TTS\data\common_voice\validated_dutch.tsv"
best_data, rankings = extract_best_speaker(tsv_dir, min_samples=1000)

🏆 Top Speaker ID: f1d68c1db286092eb50d7d911841e6bf5d7bdb63128e3d23c2c49bdf47502f5d953f248d9ce137dc914958dda945dc92ad287ee4e81abd9d7aa03d14dea19d58
📊 Total Samples: 15754
⭐ Quality Score: 30783.00


In [7]:
xtts_training_data = filter_xtts_subset(best_data, target_client_id="f1d68c1db286092eb50d7d911841e6bf5d7bdb63128e3d23c2c49bdf47502f5d953f248d9ce137dc914958dda945dc92ad287ee4e81abd9d7aa03d14dea19d58", target_samples=350)
xtts_training_data.to_csv("xtts_finetune_subset.tsv", sep='\t', index=False)

Initial samples for speaker: 15754
Samples after strict voting filter: 15391
Samples after text length filter: 15123
Randomly sampled down to 350 target samples.


In [10]:
source_dir = r"D:\AudioDataset\CV-Mozilla-25-Netherland\cv-corpus-25.0-2026-03-09\nl\clips" 
output_dir = r"D:\AudioDataset\CV-Mozilla-25-Netherland\cv-corpus-25.0-2026-03-09\nl\clips_processed"
process_and_convert_to_wav(xtts_training_data, source_dir, output_dir)

Starting conversion of 350 files to 22050Hz WAV...


Processing Audio: 100%|██████████| 350/350 [00:17<00:00, 19.53it/s]


✅ Successfully processed and converted 350 files.


350

In [13]:
generate_xtts_metadata(xtts_training_data, r"C:\Users\alvar\Code\Github\MedEmoji-TTS\data\common_voice\metadata_dutch.csv")

📝 Saved 350 training entries to: C:\Users\alvar\Code\Github\MedEmoji-TTS\data\common_voice\metadata_dutch.csv


## Chinese Dataset

In [31]:
tsv_dir = r"C:\Users\alvar\Code\Github\MedEmoji-TTS\data\common_voice\validated_chinese.tsv"
best_data, rankings = extract_best_speaker(tsv_dir, min_samples=500)

C:\Users\alvar\AppData\Local\Temp\ipykernel_2052\2744427984.py:9: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(tsv_path, sep='\t')


🏆 Top Speaker ID: 38f58268a87e9462be6cbcbc3871f9eca99b2f1eae0b6d0f388e29250a823c5435c5407e2666fc1e1b7a00231e2a9bf661bfa16b095202b7ce07145512a30b05
📊 Total Samples: 813
⭐ Quality Score: 4020.00


In [36]:
xtts_training_data = filter_xtts_subset_chinese(best_data, target_client_id="38f58268a87e9462be6cbcbc3871f9eca99b2f1eae0b6d0f388e29250a823c5435c5407e2666fc1e1b7a00231e2a9bf661bfa16b095202b7ce07145512a30b05", target_samples=500)
xtts_training_data.to_csv("xtts_finetune_subset.tsv", sep='\t', index=False)

Initial samples for speaker: 813
Samples after strict voting filter: 675
Samples after text length filter: 509
Randomly sampled down to 500 target samples.


In [37]:
source_dir = r"D:\AudioDataset\CV-Mozilla-25-Chinese\cv-corpus-25.0-2026-03-09\zh-CN\clips" 
output_dir = r"D:\AudioDataset\CV-Mozilla-25-Chinese\cv-corpus-25.0-2026-03-09\zh-CN\clips_processed"
process_and_convert_to_wav(xtts_training_data, source_dir, output_dir)

Starting conversion of 500 files to 22050Hz WAV...


Processing Audio: 100%|██████████| 500/500 [00:15<00:00, 32.31it/s]


✅ Successfully processed and converted 500 files.


500

In [40]:
generate_xtts_metadata(xtts_training_data, r"C:\Users\alvar\Code\Github\MedEmoji-TTS\data\common_voice\metadata_chinese.csv")

📝 Saved 500 training entries to: C:\Users\alvar\Code\Github\MedEmoji-TTS\data\common_voice\metadata_chinese.csv
